# 04 - Comparisons

Compares pairs of finished runs from `experiments/` using the metrics each run saved, so no model is retrained here. The pairs are listed in section 0.

Metrics are grouped in two families:
- Separation: silhouette (higher is better), Davies-Bouldin (lower is better) and Calinski-Harabasz (higher is better).
- Persistence: mean run length (higher is better) and switches per year (lower is better).

A run wins a family when it wins more of its metrics. The overall winner wins both families, or one family with the other tied. If each run wins one family, the split is resolved by total metric wins and marked as split.

Separation metrics are measured in each run's own feature space (latents + daily level). They compare cleanly only when both runs share the model and the scalogram set, for example k-means vs GMM on the same set. When the set or the model changes, use persistence, the regime profiles and the regime timelines instead. The report adds a note in those cases.

Results are saved to `experiments/comparisons.csv` (one row per pair and metric) and `experiments/comparisons_summary.csv` (one row per pair). Comparing a pair again replaces its rows.

<d.benavidess@uniandes.edu.co>

In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd

from src.comparison import compare, report, summary_table, save_comparisons

## 0. Pairs to compare

Each pair is (run A, run B), written as the run folder path inside `experiments/`. Change one thing per pair (algorithm, scalogram set or model) so the difference has a single cause.

In [2]:
pairs = [
    ("conv_vae/kmeans/001_real_price_kmeans_k2", "conv_vae/gmm/002_real_price_gmm_k2"),
    ("conv_vae/kmeans/003_real_price_log_kmeans_k2", "conv_vae/gmm/004_real_price_log_gmm_k2"),
    ("conv_vae/kmeans/001_real_price_kmeans_k2", "conv_vae/kmeans/003_real_price_log_kmeans_k2"),
    ("conv_vae/gmm/002_real_price_gmm_k2", "conv_vae/gmm/004_real_price_log_gmm_k2"),
]

DATA_DIR = Path("../data/processed")
EXPERIMENTS_DIR = Path("../experiments")

## 1. Comparisons

In [3]:
results = []
for run_a, run_b in pairs:
    result = compare(run_a, run_b, DATA_DIR, EXPERIMENTS_DIR)
    report(result)
    results.append(result)
    print("\n" + "=" * 100 + "\n")

A = conv_vae/kmeans/001_real_price_kmeans_k2
B = conv_vae/gmm/002_real_price_gmm_k2

What changes between the runs
+------------+--------+-----+
|  Setting   |   A    |  B  |
+------------+--------+-----+
| clustering | kmeans | gmm |
+------------+--------+-----+

Clustering metrics
+-------------------+-------------+--------+-----------+-----------+--------+
|       Metric      |    Group    | Better |     A     |     B     | Winner |
+-------------------+-------------+--------+-----------+-----------+--------+
|     Silhouette    |  separation | higher |   0.340   |   0.310   |   A    |
|   Davies-Bouldin  |  separation | lower  |   1.239   |   1.294   |   A    |
| Calinski-Harabasz |  separation | higher | 3,797.777 | 3,365.439 |   A    |
|  Mean run (days)  | persistence | higher |   4.425   |   4.210   |   A    |
| Switches per year | persistence | lower  |   82.500  |   86.713  |   A    |
+-------------------+-------------+--------+-----------+-----------+--------+

Regime profi

## 2. Summary

In [4]:
print(summary_table([r["summary"] for r in results]))
for r in results:
    print(f"- {r['summary']['verdict']}")
    for note in r["checks"]:
        print(f"  Note: {note}")

+----------------------------------------------+----------------------------------------------+------------+-------------+----------------+----------------+
| Run A                                        | Run B                                        | Separation | Persistence | Total wins A-B | Overall winner |
+----------------------------------------------+----------------------------------------------+------------+-------------+----------------+----------------+
| conv_vae/kmeans/001_real_price_kmeans_k2     | conv_vae/gmm/002_real_price_gmm_k2           | 3-0 (A)    | 2-0 (A)     | 5-0            | A              |
| conv_vae/kmeans/003_real_price_log_kmeans_k2 | conv_vae/gmm/004_real_price_log_gmm_k2       | 3-0 (A)    | 2-0 (A)     | 5-0            | A              |
| conv_vae/kmeans/001_real_price_kmeans_k2     | conv_vae/kmeans/003_real_price_log_kmeans_k2 | 3-0 (A)    | 0-2 (B)     | 3-2            | A              |
| conv_vae/gmm/002_real_price_gmm_k2           | conv_vae/

In [5]:
summary = save_comparisons(results, EXPERIMENTS_DIR)
summary[["date", "run_a", "run_b", "overall_winner", "basis", "changed", "same_regime_share"]]

,date,run_a,run_b,overall_winner,basis,changed,same_regime_share
0,2026-10-07T15:50:59,conv_vae/kmeans/001_real_price_kmeans_k2,conv_vae/gmm/002_real_price_gmm_k2,conv_vae/kmeans/001_real_price_kmeans_k2,groups,clustering: kmeans -> gmm,0.932125
1,2026-10-07T15:50:59,conv_vae/kmeans/003_real_price_log_kmeans_k2,conv_vae/gmm/004_real_price_log_gmm_k2,conv_vae/kmeans/003_real_price_log_kmeans_k2,groups,clustering: kmeans -> gmm,0.844062
2,2026-10-07T15:50:59,conv_vae/kmeans/001_real_price_kmeans_k2,conv_vae/kmeans/003_real_price_log_kmeans_k2,conv_vae/kmeans/001_real_price_kmeans_k2,"split groups, total wins",scalogram_set: real_price -> real_price_log,0.204449
3,2026-10-07T15:50:59,conv_vae/gmm/002_real_price_gmm_k2,conv_vae/gmm/004_real_price_log_gmm_k2,conv_vae/gmm/002_real_price_gmm_k2,groups,scalogram_set: real_price -> real_price_log,0.173447
